In [ ]:
import cv2 as cv
import math  
import numpy as np

In [ ]:
class ColorDetector:
    def __init__(self, colors=["green", "red", "blue", "yellow"]) -> None: 
        all_ranges = {
            "green": {
                "ranges": [(np.array([40, 50, 50]), np.array([80, 255, 255]))],
                "bgr": (0, 255, 0)
            },
            "blue": {
                "ranges": [(np.array([90, 50, 50]), np.array([130, 255, 255]))],
                "bgr": (255, 0, 0)
            },
            "yellow": { 
                "ranges": [(np.array([20, 50, 50]), np.array([35, 255, 255]))],
                "bgr": (0, 255, 255) 
            },
            "red": {
                "ranges": [
                    (np.array([0, 50, 50]), np.array([10, 255, 255])),
                    (np.array([170, 50, 50]), np.array([179, 255, 255]))
                ],
                "bgr": (0, 0, 255)
            }
        }

        self.kernel = np.ones((5, 5), np.uint8)
        self.colors_ranges = {color: all_ranges[color] for color in colors if color in all_ranges}
    
    def detect_leds(self, image : np.ndarray) -> tuple[np.ndarray, list[dict]]:

        annotated_images = image.copy()
        hsv = cv.cvtColor(annotated_images, cv.COLOR_BGR2HSV)
        detected_points = []

        for color_name, param in self.colors_ranges.items(): 
            full_mask = np.zeros(hsv.shape[:2], dtype=np.uint8)

            for lower, upper in param["ranges"]:
                mask = cv.inRange(hsv, lower, upper)
                full_mask = cv.bitwise_or(full_mask, mask)

            full_mask = cv.dilate(full_mask, self.kernel, iterations=2) 

            contours, _ = cv.findContours(full_mask, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)

            for cnt in contours:
                area = cv.contourArea(cnt)
                if area > 100:
                    x, y, w, h = cv.boundingRect(cnt)

                    cx = x + w // 2
                    cy = y + h // 2

                    detected_points.append({
                        "color": color_name,
                        "center": (cx, cy),
                        "bbox": (x, y, w, h), 
                        "area": area 
                    })

                    cv.rectangle(annotated_images, (x, y), (x+w, y+h), param["bgr"], 2)
                    cv.putText(annotated_images, color_name, (x, y-10), cv.FONT_HERSHEY_SIMPLEX, 0.6, param["bgr"], 2)
                    cv.circle(annotated_images, (cx, cy), 3, param["bgr"], -1)

        return annotated_images, detected_points

In [ ]:
detector = ColorDetector(colors=["green", "red", "blue", "yellow"])
cap = cv.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    processed_frame, leds_data = detector.detect_leds(frame)
    
    found_colors = {}
    for led in leds_data:
        color = led["color"]
        if color not in found_colors:
            found_colors[color] = []
        found_colors[color].append(led["center"])

    if "green" in found_colors:
        print(f"Зелений: {found_colors['green']}")
    if "red" in found_colors:
        print(f"Червоний: {found_colors['red']}")
    if "blue" in found_colors:
        print(f"Синій: {found_colors['blue']}")
    if "yellow" in found_colors: 
        print(f"Жовтий: {found_colors['yellow']}")
        
    cv.imshow("Swarm Vision", processed_frame)

    if cv.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv.destroyAllWindows()